# PyTorch tensor-shape drills

Twelve exercises on shape, reshape, broadcasting, and reductions. These are designed for Google Colab and do not need a GPU.

## How to work

1. Read an exercise without running its code.
2. Calculate the answer on paper.
3. Replace each `None` with your predicted tuple or integer.
4. Run the cell. The checker says whether your prediction matches PyTorch.
5. If it fails, explain the mismatch on paper before changing your answer.

Keep the Colab hardware accelerator set to **None** for these drills so you do not spend GPU quota unnecessarily.

In [1]:
import torch

print(f"PyTorch version: {torch.__version__}")
print("Device for these drills: CPU")

RESULTS = {}

def shape_ok(label, predicted, actual):
    if predicted is None:
        print(f"{label}: NOT ANSWERED — replace None with a tuple.")
        return False
    try:
        predicted = tuple(predicted)
    except TypeError:
        print(f"{label}: Use a tuple such as (2, 3).")
        return False
    actual = tuple(actual)
    if predicted == actual:
        print(f"{label}: PASS")
        return True
    print(f"{label}: TRY AGAIN — your prediction {predicted} does not match PyTorch.")
    return False

def value_ok(label, predicted, actual):
    if predicted is None:
        print(f"{label}: NOT ANSWERED — replace None with an integer.")
        return False
    if predicted == actual:
        print(f"{label}: PASS")
        return True
    print(f"{label}: TRY AGAIN — your prediction {predicted} does not match PyTorch.")
    return False

PyTorch version: 2.11.0+cu130
Device for these drills: CPU


## 1. Shape and rank

A tensor is created from 24 values and reshaped with dimensions 2, 3, and 4. Before running the cell, predict its shape and rank (`ndim`).

In [2]:
paper_shape = (2, 3, 4)  # Replace with a tuple
paper_rank = 3   # Replace with an integer

x = torch.arange(24).reshape(2, 3, 4)
checks = [
    shape_ok("1 shape", paper_shape, x.shape),
    value_ok("1 rank", paper_rank, x.ndim),
]
RESULTS[1] = all(checks)

1 shape: PASS
1 rank: PASS


## 2. Slicing

Start from a tensor with shape `(2, 3, 4, 5)`. Keep every item in dimension 0, keep indices 1 onward in dimension 1, keep every item in dimension 2, and keep only the first two entries in the last dimension. Predict the result shape.

In [3]:
paper_shape = (2, 2, 4, 2)

x = torch.arange(120).reshape(2, 3, 4, 5)
result = x[:, 1:, :, :2]
RESULTS[2] = shape_ok("2", paper_shape, result.shape)

2: PASS


## 3. Reshape with `-1`

A tensor has shape `(2, 3, 4)`. It is reshaped to `(2, -1)`. Calculate the inferred dimension and predict the output shape.

In [4]:
paper_shape = (2, 12)

x = torch.arange(24).reshape(2, 3, 4)
result = x.reshape(2, -1)
RESULTS[3] = shape_ok("3", paper_shape, result.shape)

3: PASS


## 4. Reshape, then permute

A tensor with 24 values is reshaped to `(2, 3, 2, 2)`, then permuted with order `(0, 2, 3, 1)`. Track where every axis moves and predict the final shape.

In [5]:
paper_shape = (2,2,2,3)

x = torch.arange(24).reshape(2, 3, 2, 2)
result = x.permute(0, 2, 3, 1)
RESULTS[4] = shape_ok("4", paper_shape, result.shape)

4: PASS


## 5. Image layout conversion

An image batch uses NCHW layout with shape `(8, 3, 32, 32)`. Convert it to NHWC with `permute(0, 2, 3, 1)`. Predict the new shape.

In [6]:
paper_shape = (8,32,32,3)

images = torch.zeros(8, 3, 32, 32)
result = images.permute(0, 2, 3, 1)
RESULTS[5] = shape_ok("5", paper_shape, result.shape)

5: PASS


## 6. `unsqueeze` and `squeeze`

Start with shape `(5, 7)`. Add a dimension at the front, add another at the end, then remove dimension 0. Predict the final shape.

In [7]:
paper_shape = (5,7,1)

x = torch.zeros(5, 7)
result = x.unsqueeze(0).unsqueeze(-1).squeeze(0)
RESULTS[6] = shape_ok("6", paper_shape, result.shape)

6: PASS


## 7. `stack` versus `cat`

Two tensors each have shape `(3, 4)`. Predict both results:

- `torch.stack([a, b], dim=0)`
- `torch.cat([a, b], dim=1)`

In [8]:
paper_stack_shape = (2,3,4)
paper_cat_shape = (3,8)

a = torch.zeros(3, 4)
b = torch.ones(3, 4)
stacked = torch.stack([a, b], dim=0)
concatenated = torch.cat([a, b], dim=1)
checks = [
    shape_ok("7 stack", paper_stack_shape, stacked.shape),
    shape_ok("7 cat", paper_cat_shape, concatenated.shape),
]
RESULTS[7] = all(checks)

7 stack: PASS
7 cat: PASS


## 8. Basic broadcasting

A matrix has shape `(4, 3)` and a bias vector has shape `(3,)`. They are added. Align dimensions from the right and predict the result shape.

In [12]:
paper_shape = (4, 3)

matrix = torch.zeros(4, 3)
bias = torch.arange(3)
result = matrix + bias
RESULTS[8] = shape_ok("8", paper_shape, result.shape)

8: PASS


## 9. Broadcasting across an image batch

Images have shape `(8, 3, 32, 32)`. Channel means have shape `(3, 1, 1)`. Subtract the means from the images and predict the output shape. Also write the implicit four-dimensional shape used for the means during broadcasting.

In [13]:
paper_output_shape = (8, 3, 32, 32)
paper_implicit_mean_shape = (1, 3, 1, 1)

images = torch.randn(8, 3, 32, 32)
channel_means = torch.tensor([0.5, 0.4, 0.3]).reshape(3, 1, 1)
result = images - channel_means
implicit_mean_shape = (1,) + tuple(channel_means.shape)
checks = [
    shape_ok("9 output", paper_output_shape, result.shape),
    shape_ok("9 implicit means", paper_implicit_mean_shape, implicit_mean_shape),
]
RESULTS[9] = all(checks)

9 output: PASS
9 implicit means: PASS


## 10. Batched matrix multiplication

The left tensor has shape `(5, 2, 3)` and the right tensor has shape `(1, 3, 4)`. `torch.matmul` broadcasts the batch dimension and multiplies the last two dimensions. Predict the result shape.

In [14]:
paper_shape = (5, 2, 4)

left = torch.randn(5, 2, 3)
right = torch.randn(1, 3, 4)
result = torch.matmul(left, right)
RESULTS[10] = shape_ok("10", paper_shape, result.shape)

10: PASS


## 11. Reduction with and without `keepdim`

A tensor has shape `(2, 3, 4)`. Predict the shapes of:

- `x.sum(dim=1)`
- `x.sum(dim=1, keepdim=True)`

In [15]:
paper_without_keepdim = (2, 4)
paper_with_keepdim = (2, 1, 4)

x = torch.randn(2, 3, 4)
without_keepdim = x.sum(dim=1)
with_keepdim = x.sum(dim=1, keepdim=True)
checks = [
    shape_ok("11 without keepdim", paper_without_keepdim, without_keepdim.shape),
    shape_ok("11 with keepdim", paper_with_keepdim, with_keepdim.shape),
]
RESULTS[11] = all(checks)

11 without keepdim: PASS
11 with keepdim: PASS


## 12. Multi-axis reduction and `argmax`

A tensor has shape `(2, 3, 4)`. Predict the shapes of:

- `x.mean(dim=(0, 2), keepdim=True)`
- `x.argmax(dim=1)`

For each operation, mark which axes disappear and which remain.

In [31]:
paper_mean_shape = (1, 3, 1)
paper_argmax_shape = (2, 4)

x = torch.randn(2, 3, 4)
mean_result = x.mean(dim=(0, 2), keepdim=True)
argmax_result = x.argmax(dim=1)
checks = [
    shape_ok("12 mean", paper_mean_shape, mean_result.shape),
    shape_ok("12 argmax", paper_argmax_shape, argmax_result.shape),
]
RESULTS[12] = all(checks)

12 mean: PASS
12 argmax: PASS


## Final check and reflection

Run the cell below after attempting all twelve drills. Do not move on merely because you found a tuple that passes. For every corrected answer, write one sentence explaining the rule you initially missed.

In [32]:
missing = [number for number in range(1, 13) if number not in RESULTS]
passed = sum(RESULTS.get(number, False) for number in range(1, 13))

print(f"Passed: {passed}/12 drills")
if missing:
    print(f"Cells not run yet: {missing}")
elif passed == 12:
    print("All drills passed. Save the notebook as your evidence.")
else:
    print("Return to every TRY AGAIN or NOT ANSWERED result before continuing.")

Passed: 12/12 drills
All drills passed. Save the notebook as your evidence.
